# M3L2 E06 - FAISS: guardar vectores y buscar por similitud

## Un solo concepto

En E05 calculamos vectores y comparamos similitudes a mano.
Con 5 textos es manejable. Con 50.000 textos, es imposible comparar uno por uno.

**FAISS** (Facebook AI Similarity Search) es una libreria que almacena vectores
y encuentra los mas similares de forma eficiente.

LangChain envuelve FAISS en `FAISS` (de `langchain_community.vectorstores`)
que acepta textos directamente y maneja los embeddings internamente.

## Necesita OpenAI API key y faiss-cpu


In [ ]:
# !pip install faiss-cpu  # descomenta si necesitas instalar

import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
print("Listo.")


## FAISS: de la busqueda naive al vector store (Lecture M3L2 - Seccion 13)

En el script legacy de M3L2 E02 haciamos:

```python
# Busqueda naive: manda TODO el contexto siempre
context = " ".join(DOCS_EMPRESA)  # todos los documentos concatenados
```

FAISS reemplaza eso con busqueda por similitud de vectores:

```text
Sin FAISS (naive)                  Con FAISS (vector store)
---------------------------------  ----------------------------------
context = join(TODOS_LOS_DOCS)     query_vector = embed("vacaciones")
                                   top_k = faiss.search(query_vector, k=2)
                                   context = format(top_k)

Manda 5 docs aunque solo           Solo manda los 2 docs mas similares
1 sea relevante                    a la pregunta
```

**Por que esto importa** (Lecture M3L2 - Seccion 13.3):

> "Si el retrieval esta encapsulado, puedes cambiar la tecnologia sin romper el resto.
> El pipeline sigue usando la interfaz Retriever."

> -- Lecture M3L2, Seccion 13.3

Hoy: `FAISS` (en memoria, sin dependencias)
Manana: `Chroma` (persistente en disco)
Pasado: `Pinecone` (en la nube, escalable a millones)

El pipeline no cambia. Solo cambia la linea que crea el vector store.


## Los documentos

Estos son los textos que vamos a indexar. En produccion vendrian de archivos o una DB.


In [ ]:
DOCS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "El seguro medico esta incluido desde el primer dia de trabajo.",
    "El horario es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta permitido 3 dias por semana.",
    "Los bonos anuales se pagan en diciembre segun desempeno.",
    "Para solicitar vacaciones hay que completar el formulario HR-01.",
]

print(f"{len(DOCS)} documentos disponibles para indexar.")


## TODO 1: crear el vector store

`FAISS.from_texts(textos, embeddings)` hace todo en un paso:
1. Llama a `embeddings.embed_documents(textos)` internamente.
2. Almacena los vectores en un indice FAISS.
3. Devuelve el objeto listo para busquedas.


In [ ]:
# TODO 1: crear el vector store con FAISS.from_texts(DOCS, embeddings)
vectorstore = None  # reemplazar

print(f"Vector store: {type(vectorstore).__name__ if vectorstore else 'TODO no completado'}")


## TODO 2: busqueda directa con similarity_search

`vectorstore.similarity_search(query, k=N)` busca los N documentos mas similares.

Devuelve una lista de `Document` con `.page_content` y `.metadata`.


In [ ]:
# TODO 2: buscar los 2 documentos mas similares a "vacaciones"
# docs = vectorstore.similarity_search("vacaciones", k=2)

# Descomentar despues de completar TODO 1 y 2:
# print(f"Documentos encontrados para 'vacaciones': {len(docs)}")
# for i, doc in enumerate(docs):
#     print(f"  Doc {i+1}: {doc.page_content}")


## TODO 3: comparar dos queries distintas

El vector store devuelve documentos diferentes segun la query.
Esto es lo que hace al retrieval mejor que "mandar todo el contexto".


In [ ]:
# TODO 3: buscar con dos queries distintas y ver que los resultados son diferentes
queries = ["dias de vacaciones", "trabajo desde casa", "pago de bonos"]

# for query in queries:
#     docs = vectorstore.similarity_search(query, k=1)
#     print(f"Query: '{query}'")
#     print(f"  Resultado: {docs[0].page_content}")
#     print()


In [ ]:
def run_checks():
    assert vectorstore is not None, "TODO 1: vectorstore es None"
    docs = vectorstore.similarity_search("vacaciones", k=2)
    assert len(docs) == 2, "k=2 debe devolver exactamente 2 documentos"
    assert hasattr(docs[0], 'page_content')
    contenidos = [d.page_content for d in docs]
    assert any("vacaciones" in c.lower() or "15 dias" in c.lower() for c in contenidos), \
        "El doc mas relevante para 'vacaciones' debe ser sobre vacaciones"
    # Con k=1 para 'remoto' debe traer el doc de trabajo remoto
    docs_remoto = vectorstore.similarity_search("trabajo desde casa", k=1)
    assert "remoto" in docs_remoto[0].page_content.lower()
    print("M3L2 E06 checks passed")

run_checks()


## Cierre

| Concepto | Que hace |
|---|---|
| `FAISS.from_texts(docs, emb)` | Crea el indice: embede los textos y los guarda |
| `vectorstore.similarity_search(q, k)` | Busca los k documentos mas similares a q |
| `doc.page_content` | El texto del documento recuperado |
| Por que importa | No mandamos TODO el contexto: solo los documentos relevantes |

**Siguiente**: E07 muestra como `as_retriever()` encapsula esta busqueda
en una interfaz estandar que se conecta a la chain.
